<a href="https://colab.research.google.com/github/ItsUgesh/wine-type-svm/blob/main/Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from dash import Dash
from dash import dcc,html,Input, Output
import yfinance as yf
import plotly.express as px
import pandas as pd

In [9]:
app = Dash(__name__)
app.layout = html.Div([
    dcc.Dropdown(
        id='stock-selector',
        options=[
            {'label': 'Apple', 'value': 'AAPL'},
            {'label': 'Microsoft', 'value': 'MSFT'},
            {'label': 'Amazon', 'value': 'AMZN'},
            {'label': 'Tesla', 'value': 'TSLA'}

        ],
        multi = True,
        value='AAPL'
    ),
    dcc.Graph(id='stock-price-chart')
])

In [12]:
from dash import Dash, dcc, html, dash_table, Input, Output
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]
START = "2022-01-01"
MA_WINDOWS = [20, 50]


def tidy_close(raw, tickers):
    """yfinance table -> one row per Date and Ticker, with a Close column."""
    close = raw["Close"]
    if isinstance(close, pd.Series):
        close = close.to_frame(tickers[0])
    close.index.name = "Date"
    return close.reset_index().melt(id_vars="Date", var_name="Ticker", value_name="Close").dropna()


def load_prices(tickers, start):
    try:
        raw = yf.download(tickers, start=start, auto_adjust=True, progress=False)
        df = tidy_close(raw, tickers)
        if df.empty:
            raise ValueError("yfinance returned no data")
    except Exception as e:
        print("yfinance failed, reading /content/stock_data.csv instead:", e)
        df = pd.read_csv("/content/stock_data.csv", parse_dates=["Date"])[["Date", "Ticker", "Close"]]
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
    for w in MA_WINDOWS:   # moving averages use the full history, so they are valid at the start of any window
        df[f"MA{w}"] = df.groupby("Ticker")["Close"].transform(lambda s: s.rolling(w).mean())
    return df


prices = load_prices(TICKERS, START)
print(prices["Ticker"].unique(), prices["Date"].min().date(), "to", prices["Date"].max().date(), len(prices), "rows")
prices.head()

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AAPL']: OperationalError('database is locked')


['AMZN' 'GOOGL' 'MSFT' 'TSLA'] 2022-01-03 to 2026-10-29 4844 rows


,Date,Ticker,Close,MA20,MA50
0,2022-01-03,AMZN,170.404495,NaN,NaN
1,2022-01-04,AMZN,167.522003,NaN,NaN
2,2022-01-05,AMZN,164.356995,NaN,NaN
3,2022-01-06,AMZN,163.253998,NaN,NaN
4,2022-01-07,AMZN,162.554001,NaN,NaN


In [13]:
STAT_COLS = ["Ticker", "Start", "Latest", "Change %", "High", "Low", "Daily volatility %"]
COLORS = px.colors.qualitative.Plotly


def filter_data(selected, start_date, end_date):
    d = prices[prices["Ticker"].isin(selected or [])]
    return d[(d["Date"] >= pd.to_datetime(start_date)) & (d["Date"] <= pd.to_datetime(end_date))]


def empty_figure(message):
    fig = go.Figure()
    fig.update_layout(annotations=[dict(text=message, showarrow=False, font=dict(size=16))],
                      xaxis_visible=False, yaxis_visible=False, template="plotly_white")
    return fig


def make_stats(d):
    rows = []
    for ticker, g in d.groupby("Ticker"):
        first, last = g["Close"].iloc[0], g["Close"].iloc[-1]
        rows.append({
            "Ticker": ticker,
            "Start": round(first, 2),
            "Latest": round(last, 2),
            "Change %": round((last / first - 1) * 100, 2),
            "High": round(g["Close"].max(), 2),
            "Low": round(g["Close"].min(), 2),
            "Daily volatility %": round(g["Close"].pct_change().std() * 100, 2),
        })
    return rows


app = Dash(__name__)
min_date, max_date = prices["Date"].min(), prices["Date"].max()

app.layout = html.Div(style={"maxWidth": "1100px", "margin": "0 auto", "fontFamily": "sans-serif"}, children=[
    html.H2("Stock Price Analysis Dashboard"),

    html.Div(style={"display": "flex", "gap": "24px", "flexWrap": "wrap", "alignItems": "end"}, children=[
        html.Div(style={"minWidth": "320px", "flex": "1"}, children=[
            html.Label("Stocks"),
            dcc.Dropdown(id="stock-selector",
                         options=[{"label": t, "value": t} for t in sorted(prices["Ticker"].unique())],
                         value=sorted(prices["Ticker"].unique())[:2], multi=True),
        ]),
        html.Div(children=[
            html.Label("Date range"),
            html.Br(),
            dcc.DatePickerRange(id="date-range", min_date_allowed=min_date, max_date_allowed=max_date,
                                start_date=max(min_date, max_date - pd.Timedelta(days=365)),
                                end_date=max_date),
        ]),
        html.Div(children=[
            html.Label("Moving averages"),
            dcc.Checklist(id="ma-options",
                          options=[{"label": f" {w}-day", "value": w} for w in MA_WINDOWS],
                          value=[20], inline=True),
        ]),
    ]),

    html.H4("Summary for the selected period"),
    dash_table.DataTable(id="stats-table", columns=[{"name": c, "id": c} for c in STAT_COLS], data=[],
                         style_cell={"textAlign": "center", "padding": "6px"},
                         style_header={"fontWeight": "bold"},
                         style_data_conditional=[
                             {"if": {"filter_query": "{Change %} < 0", "column_id": "Change %"}, "color": "#c0392b"},
                             {"if": {"filter_query": "{Change %} >= 0", "column_id": "Change %"}, "color": "#1e8449"}]),

    dcc.Graph(id="stock-price-chart"),
    dcc.Graph(id="returns-chart"),
])


@app.callback(
    Output("stock-price-chart", "figure"),
    Output("returns-chart", "figure"),
    Output("stats-table", "data"),
    Input("stock-selector", "value"),
    Input("date-range", "start_date"),
    Input("date-range", "end_date"),
    Input("ma-options", "value"),
)
def update_dashboard(selected_stocks, start_date, end_date, ma_windows):
    d = filter_data(selected_stocks, start_date, end_date)
    if d.empty:
        msg = "Select at least one stock and a date range with data"
        return empty_figure(msg), empty_figure(msg), []

    price_fig = go.Figure()
    for i, (ticker, g) in enumerate(d.groupby("Ticker")):
        color = COLORS[i % len(COLORS)]
        price_fig.add_trace(go.Scatter(x=g["Date"], y=g["Close"], name=ticker, mode="lines",
                                       line=dict(color=color, width=2), legendgroup=ticker))
        for w in ma_windows or []:
            price_fig.add_trace(go.Scatter(x=g["Date"], y=g[f"MA{w}"], name=f"{ticker} {w}-day MA", mode="lines",
                                           line=dict(color=color, width=1, dash="dot"), legendgroup=ticker))
    price_fig.update_layout(title="Closing price", yaxis_title="Price (USD)", template="plotly_white",
                            hovermode="x unified")

    rebased = d.copy()
    rebased["Index"] = rebased.groupby("Ticker")["Close"].transform(lambda s: s / s.iloc[0] * 100)
    returns_fig = px.line(rebased, x="Date", y="Index", color="Ticker", template="plotly_white",
                          title="Performance: every stock rebased to 100 at the start of the period")
    returns_fig.add_hline(y=100, line_dash="dash", line_color="grey")
    returns_fig.update_layout(yaxis_title="Index (start = 100)", hovermode="x unified")

    return price_fig, returns_fig, make_stats(d)

In [15]:
app.run(debug=False, jupyter_height=1100)

<IPython.core.display.Javascript object>